## Load Dataset

In [39]:
import pandas as pd
import onnxruntime as ort
import onnx
import numpy as np
import dice_ml
from dice_ml import Dice
# cf_df = pd.read_csv("all_counterfactuals.csv")
# cf_df.head()

In [3]:
# load training and testing data
training_data = pd.read_csv("for_testing/training_data_2021-11-23 00:00:00.csv")
testing_data = pd.read_csv("for_testing/testing_data_2021-11-23 00:00:00.csv")
print(training_data.loc[43424])

past_profitability_21d       0.025818
past_profitability_63d       0.046051
past_profitability_126d      0.131742
volatility_21d               0.078286
volatility_63d               0.079081
volatility_126d              0.078615
avg_price_21d                8.400078
avg_price_63d                8.368175
avg_price_126d               8.056539
sharpe_21d                   0.329959
sharpe_63d                   0.582737
sharpe_126d                  1.676071
m_21d                         0.21452
m_63d                         0.37516
m_126d                        0.99222
roc_21d                      0.025818
roc_63d                      0.046051
roc_126d                     0.131742
MACD                          0.05093
rsi_14                      63.016488
dco_22                      -0.053505
min_21d                        8.2331
min_63d                      8.146272
min_126d                       7.5163
max_21d                        8.5652
max_63d                        8.5652
max_126d    

In [16]:
training_data = training_data.drop(columns=["col_timestamp"])
testing_data = testing_data.drop(columns=["col_timestamp"])

In [26]:
# load trained model
session = ort.InferenceSession("for_testing/profitability_recommendation_2021-11-23 00:00:00.onnx")

model = onnx.load("for_testing/profitability_recommendation_2021-11-23 00:00:00.onnx")
onnx.checker.check_model(model)

In [31]:
input_name = session.get_inputs()[0].name
label_name = session.get_outputs()[0].name
input_name, label_name

('float_input', 'variable')

In [211]:
query_to_predict = testing_data.drop(columns="target")[0:1].values.astype(np.float32)
print(query_to_predict)

[[-7.38290325e-02  2.13770211e-01  2.58344769e-01  3.10262740e-01
   4.33814436e-01  4.05843347e-01  4.95974493e+00  4.71617842e+00
   4.14708900e+00 -2.36372441e-01  4.93708938e-01  6.37239993e-01
  -3.75000000e-01  8.24999988e-01  9.59999979e-01 -7.38290325e-02
   2.13770211e-01  2.58344769e-01 -3.37277204e-02  3.66351509e+01
   1.23516366e-01  4.61399984e+00  3.91300011e+00  3.31999993e+00
   5.19000006e+00  5.19000006e+00  5.19000006e+00  4.86837721e+00
   4.64618778e+00  4.23288155e+00]]


In [212]:
pred_onx = session.run(None, {input_name: query_to_predict})[0]

In [213]:
pred_onx

array([[0.00463372]], dtype=float32)

## Dice

Works on regression, but it takes too long to generate each counterfactual. CARLA is optimized but only works on classification.

In [214]:


y = training_data["target"]

X = training_data.drop("target", axis=1)

continuous_cols = X.select_dtypes(include=["number"]).columns.tolist()

dice_data = dice_ml.Data(
    dataframe=training_data,
    continuous_features=list(continuous_cols),
    outcome_name="target"
)



In [215]:
y.min(), y.max()

(np.float64(-0.9272918124562632), np.float64(11.238508633824596))

In [227]:
class ONNXWrapper:
    debug_calls = 0
    def __init__(self, session, feature_names):
        self.session = session
        self.input_name = session.get_inputs()[0].name
        self.feature_names = feature_names

    def predict(self, X):
        if ONNXWrapper.debug_calls < 10:
            print("DEBUG", ONNXWrapper.debug_calls, type(X), ...)
        ONNXWrapper.debug_calls += 1
        # --- 1) If X is a list of dicts (DiCE sometimes uses this)
        if isinstance(X, list) and isinstance(X[0], dict):
            X = pd.DataFrame(X)[self.feature_names]

        # --- 2) If X is a DataFrame
        if isinstance(X, pd.DataFrame):
            X = X[self.feature_names].astype(np.float32).values

        # --- 3) If X is a list or array
        else:
            X = np.array(X, dtype=np.float32)

        # --- 4) Ensure 2D
        if X.ndim == 1:
            X = X.reshape(1, -1)

        # --- 5) Run ONNX inference
        pred = self.session.run(None, {self.input_name: X})[0]

        # flatten ONNX output so DiCE returns scalars, not lists
        pred = pred.reshape(-1)  

        return pred


In [228]:
onnx_model = ONNXWrapper(session, X.columns)

In [229]:
# One instance
query_instance = testing_data.drop(columns="target")[0:1].astype(np.float32)
print("Factual instance:")
print(query_instance)

Factual instance:
   past_profitability_21d  past_profitability_63d  past_profitability_126d  \
0               -0.073829                 0.21377                 0.258345   

   volatility_21d  volatility_63d  volatility_126d  avg_price_21d  \
0        0.310263        0.433814         0.405843       4.959745   

   avg_price_63d  avg_price_126d  sharpe_21d  ...    dco_22  min_21d  min_63d  \
0       4.716178        4.147089   -0.236372  ...  0.123516    4.614    3.913   

   min_126d  max_21d  max_63d  max_126d  exp_mean_21d  exp_mean_63d  \
0      3.32     5.19     5.19      5.19      4.868377      4.646188   

   exp_mean_126d  
0       4.232882  

[1 rows x 30 columns]


In [230]:
# Sklearn wrapping
dice_model = dice_ml.Model(model=onnx_model, backend="sklearn", model_type="regressor")

# Generate counterfactual with DiCE
exp = Dice(dice_data, dice_model, method="genetic")  # "random" ou "genetic" ou "kd"

# Generate 3 counterfactuals
dice_exp = exp.generate_counterfactuals(query_instance, total_CFs=2, desired_range=[float(y.min()), float(y.max())])

# Visualisation
dice_exp.visualize_as_dataframe(show_only_changes=True)


  0%|          | 0/1 [00:00<?, ?it/s]

DEBUG 0 <class 'pandas.core.frame.DataFrame'> Ellipsis
DEBUG 1 <class 'pandas.core.frame.DataFrame'> Ellipsis


100%|██████████| 1/1 [00:00<00:00,  1.25it/s]

DEBUG 2 <class 'pandas.core.frame.DataFrame'> Ellipsis
DEBUG 3 <class 'pandas.core.frame.DataFrame'> Ellipsis
DEBUG 4 <class 'pandas.core.frame.DataFrame'> Ellipsis
DEBUG 5 <class 'pandas.core.frame.DataFrame'> Ellipsis
DEBUG 6 <class 'pandas.core.frame.DataFrame'> Ellipsis
DEBUG 7 <class 'pandas.core.frame.DataFrame'> Ellipsis
DEBUG 8 <class 'pandas.core.frame.DataFrame'> Ellipsis
DEBUG 9 <class 'pandas.core.frame.DataFrame'> Ellipsis
Query instance (original outcome : 0.00463371817022562)


,past_profitability_21d,past_profitability_63d,past_profitability_126d,volatility_21d,volatility_63d,volatility_126d,avg_price_21d,avg_price_63d,avg_price_126d,sharpe_21d,...,min_21d,min_63d,min_126d,max_21d,max_63d,max_126d,exp_mean_21d,exp_mean_63d,exp_mean_126d,target
0,-0.073829,0.21377,0.258345,0.310263,0.433814,0.405843,4.959745,4.716178,4.147089,-0.236372,...,4.614,3.913,3.32,5.19,5.19,5.19,4.868377,4.646188,4.232882,0.004634



Diverse Counterfactual set (new outcome: [-0.9272918124562632, 11.238508633824596])


,past_profitability_21d,past_profitability_63d,past_profitability_126d,volatility_21d,volatility_63d,volatility_126d,avg_price_21d,avg_price_63d,avg_price_126d,sharpe_21d,...,min_21d,min_63d,min_126d,max_21d,max_63d,max_126d,exp_mean_21d,exp_mean_63d,exp_mean_126d,target
0,-0.10000000149011612,0.20000000298023224,0.20000000298023224,0.30000001192092896,0.4000000059604645,0.4000000059604645,5.0,4.699999809265137,4.099999904632568,-0.20000000298023224,...,4.599999904632568,3.9000000953674316,3.299999952316284,5.199999809265137,5.199999809265137,5.199999809265137,4.900000095367432,4.639999866485596,4.230000019073486,-0.04089638218283653
0,0.0,0.30000001192092896,0.20000000298023224,0.4000000059604645,0.4000000059604645,0.4000000059604645,5.0,4.599999904632568,4.099999904632568,0.10000000149011612,...,4.599999904632568,3.700000047683716,3.299999952316284,5.199999809265137,5.199999809265137,5.199999809265137,5.0,4.630000114440918,4.190000057220459,-0.06193215027451515


In [231]:
cfs = dice_exp.cf_examples_list[0].final_cfs_df

In [108]:
cfs

,past_profitability_21d,past_profitability_63d,past_profitability_126d,volatility_21d,volatility_63d,volatility_126d,avg_price_21d,avg_price_63d,avg_price_126d,sharpe_21d,...,min_21d,min_63d,min_126d,max_21d,max_63d,max_126d,exp_mean_21d,exp_mean_63d,exp_mean_126d,target
0,-0.1,0.2,0.2,0.3,0.4,0.4,5.0,4.7,4.1,-0.2,...,4.6,3.9,3.3,5.2,5.2,5.2,4.9,4.64,4.23,-0.040896
0,0.0,0.3,0.2,0.4,0.4,0.4,5.0,4.6,4.1,0.1,...,4.6,3.7,3.3,5.2,5.2,5.2,5.0,4.63,4.19,-0.061932


In [243]:
#query_instance = X_test.iloc[0].values
counterfactual = cfs.drop(columns="target")[:1].values.astype(np.float32)


In [244]:
counterfactual

array([[-0.1 ,  0.2 ,  0.2 ,  0.3 ,  0.4 ,  0.4 ,  5.  ,  4.7 ,  4.1 ,
        -0.2 ,  0.5 ,  0.6 , -0.3 ,  0.9 ,  0.9 , -0.1 ,  0.2 ,  0.2 ,
        -0.  , 35.1 ,  0.1 ,  4.6 ,  3.9 ,  3.3 ,  5.2 ,  5.2 ,  5.2 ,
         4.9 ,  4.64,  4.23]], dtype=float32)

In [245]:
session.run(None, {input_name: counterfactual})[0]

array([[-0.02516202]], dtype=float32)

## Generate all counterfactuals

In [247]:
print(training_data["target"].min(),training_data["target"].max())

-0.9272918124562632 11.238508633824596


In [248]:
print(testing_data["target"].min(),testing_data["target"].max())

-0.9829017264276227 4.541095890410959


In [ ]:
all_cf = []

n = len(testing_data.iloc[:10,:])

min, max = training_data["target"].min(), training_data["target"].max()

for i in range(n):
    query_instance = testing_data.drop(columns="target").iloc[i:i+1].astype(np.float64)
    factual_id = query_instance.index[0]

    dice_exp = exp.generate_counterfactuals(
        query_instance,
        total_CFs=1,
        desired_range=[min, max]
    )
    
    cf_df = dice_exp.cf_examples_list[0].final_cfs_df.copy()
    cf_df["factual_id"] = factual_id  # rattache à son factuel

    all_cf.append(cf_df)

# Concaténer tous les contre-factuels
all_cf_df = pd.concat(all_cf, ignore_index=True)

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  1.52it/s]


In [257]:
all_cf_df

,past_profitability_21d,past_profitability_63d,past_profitability_126d,volatility_21d,volatility_63d,volatility_126d,avg_price_21d,avg_price_63d,avg_price_126d,sharpe_21d,...,min_63d,min_126d,max_21d,max_63d,max_126d,exp_mean_21d,exp_mean_63d,exp_mean_126d,target,factual_id
0,0.0,0.3,0.2,0.4,0.4,0.4,5.0,4.6,4.1,0.1,...,3.7,3.3,5.2,5.2,5.2,5.0,4.63,4.19,-0.061932,0
1,0.0,0.3,0.2,0.4,0.4,0.4,5.0,4.6,4.1,0.1,...,3.7,3.3,5.2,5.2,5.2,5.0,4.63,4.19,-0.061932,1
2,0.0,0.3,0.2,0.4,0.4,0.4,5.0,4.6,4.1,0.1,...,3.7,3.3,5.2,5.2,5.2,5.0,4.63,4.19,-0.061932,2
3,0.0,0.4,0.2,0.4,0.4,0.4,5.0,4.6,4.1,0.1,...,3.6,3.3,5.2,5.2,5.2,5.0,4.63,4.18,-0.072740,3
4,-0.1,0.1,0.2,0.3,0.4,0.3,4.7,4.4,4.1,-0.2,...,4.0,3.6,4.9,4.9,4.9,4.7,4.45,4.19,0.116457,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,-0.0,0.1,0.1,0.4,0.4,0.4,2.5,2.4,2.3,-0.0,...,2.2,2.0,2.6,2.7,2.7,2.5,2.43,2.36,0.076294,495
496,-0.0,0.1,0.1,0.4,0.4,0.4,2.5,2.4,2.3,-0.1,...,2.2,2.0,2.6,2.7,2.7,2.5,2.42,2.35,0.081662,496
497,-0.0,0.1,-0.0,0.3,0.3,0.4,2.6,2.7,2.6,-0.1,...,2.4,2.3,2.7,2.9,2.9,2.6,2.64,2.59,0.203605,497
498,-0.0,0.1,-0.0,0.3,0.3,0.4,2.6,2.7,2.6,-0.1,...,2.4,2.3,2.7,2.9,2.9,2.6,2.64,2.59,0.203605,498


In [ ]:
# Sauvegarder en CSV
# all_cf_df.to_csv("counterfactuals_500.csv", index=False)
# print("Saved counterfactuals shape:", all_cf_df.shape)

Saved counterfactuals shape: (500, 32)


In [140]:
from termcolor import colored

def highlight_changes(factual, counterfactual):
    """Affiche chaque feature, en rouge si modifiée"""
    for col in factual.index:
        f_val = factual[col]
        c_val = counterfactual[col]
        if f_val != c_val:
            print(col, ":", colored(f_val, "green"), "→", colored(c_val, "red"))
        else:
            print(col, ":", f_val)
    print("-" * 40)


In [141]:
# Exemple : visualiser pour le premier factuel
factual_id = all_cf_df["factual_id"].iloc[0]
factual = testing_data.loc[factual_id]

subset = all_cf_df[all_cf_df["factual_id"] == factual_id]

for idx, row in subset.iterrows():
    print(f"\nCounterfactual {idx}:")
    highlight_changes(factual, row.drop("factual_id"))


Counterfactual 0:
past_profitability_21d : -0.0738290311921911 → 0.0
past_profitability_63d : 0.213770214095279 → 0.30000001192092896
past_profitability_126d : 0.2583447661125139 → 0.20000000298023224
volatility_21d : 0.3102627523423307 → 0.4000000059604645
volatility_63d : 0.4338144270914076 → 0.4000000059604645
volatility_126d : 0.4058433499915637 → 0.4000000059604645
avg_price_21d : 4.959744761904762 → 5.0
avg_price_63d : 4.7161784126984125 → 4.599999904632568
avg_price_126d : 4.147089206349207 → 4.099999904632568
sharpe_21d : -0.236372434043557 → 0.10000000149011612
sharpe_63d : 0.4937089261175924 → 0.800000011920929
sharpe_126d : 0.6372400189046246 → 0.6000000238418579
m_21d : -0.3750000000000001 → 0.20000000298023224
m_63d : 0.825 → 1.2000000476837158
m_126d : 0.9599999999999996 → 0.8999999761581421
roc_21d : -0.0738290311921911 → 0.0
roc_63d : 0.213770214095279 → 0.30000001192092896
roc_126d : 0.2583447661125139 → 0.20000000298023224
MACD : -0.033727719892717 → 0.0
rsi_14 : 36.

## Evaluation

In [170]:
from typing import List, Tuple, Union

def l0_distance(delta: np.ndarray) -> List[float]:
    """
    Computes L-0 norm, number of non-zero entries.

    Parameters
    ----------
    delta: np.ndarray
        Difference between factual and counterfactual

    Returns
    -------
    List[float]
    """
    # get mask that selects all elements that are NOT zero (with some small tolerance)
    difference_mask = np.invert(np.isclose(delta, np.zeros_like(delta), atol=1e-05))
    # get the number of changed features for each row
    num_feature_changes = np.sum(
        difference_mask,
        axis=1,
        dtype=float,
    )
    distance = num_feature_changes.tolist()
    return distance

In [197]:
def l1_distance(delta: np.ndarray) -> List[float]:
    """
    Computes L-1 distance, sum of absolute difference.

    Parameters
    ----------
    delta: np.ndarray
        Difference between factual and counterfactual

    Returns
    -------
    List[float]
    """
    absolute_difference = np.abs(delta)
    distance = np.sum(absolute_difference, axis=1, dtype=float).tolist()
    return distance

In [ ]:
def l2_distance(delta: np.ndarray) -> List[float]:
    """
    Computes L-2 distance, sum of squared difference - Euclidean distance.

    Parameters
    ----------
    delta: np.ndarray
        Difference between factual and counterfactual

    Returns
    -------
    List[float]
    """
    squared_difference = np.square(np.abs(delta))
    distance = np.sum(squared_difference, axis=1, dtype=float).tolist()
    return distance


In [171]:
def _get_delta(factual: np.ndarray, counterfactual: np.ndarray) -> np.ndarray:
    """
    Compute difference between original factual and counterfactual

    Parameters
    ----------
    factual: np.ndarray
        Normalized and encoded array with factual data.
        Shape: NxM
    counterfactual: : np.ndarray
        Normalized and encoded array with counterfactual data.
        Shape: NxM

    Returns
    -------
    np.ndarray
    """
    return counterfactual - factual

In [262]:
def _get_distances(
    factual: np.ndarray, counterfactual: np.ndarray
) -> List[List[float]]:
    """
    Computes distances.
    All features have to be in the same order (without target label).

    Parameters
    ----------
    factual: np.ndarray
        Normalized and encoded array with factual data.
        Shape: NxM
    counterfactual: np.ndarray
        Normalized and encoded array with counterfactual data
        Shape: NxM

    Returns
    -------
    list: distances 1 to 4
    """
    if factual.shape != counterfactual.shape:
        raise ValueError("Shapes of factual and counterfactual have to be the same")
    if len(factual.shape) != 2:
        raise ValueError(
            "Shapes of factual and counterfactual have to be 2-dimensional"
        )

    # get difference between original and counterfactual
    delta = _get_delta(factual, counterfactual)

    #d0 = l0_distance(delta)
    #d1 = l1_distance(delta)
    d2 = l2_distance(delta)

    return [[d2[i]] for i in range(len(d2))]

In [263]:
from abc import ABC, abstractmethod

import pandas as pd


class Evaluation(ABC):
    def __init__(self, mlmodel, hyperparameters: dict = None):
        """

        Parameters
        ----------
        mlmodel:
            Classification model. (optional)
        hyperparameters:
            Dictionary with hyperparameters, could be used to pass other things. (optional)
        """
        self.mlmodel = mlmodel
        self.hyperparameters = hyperparameters

    @abstractmethod
    def get_evaluation(
        self, factuals: pd.DataFrame, counterfactuals: pd.DataFrame
    ) -> pd.DataFrame:
        """Compute evaluation measure"""

In [264]:
def remove_nans(
    counterfactuals: pd.DataFrame, factuals: pd.DataFrame = None
) -> Union[Tuple[pd.DataFrame, pd.DataFrame], pd.DataFrame]:
    """Remove instances for which a counterfactual could not be found.

    Parameters
    ----------
    counterfactuals:
        Has to be the same shape as factuals.
    factuals:
        Has to be the same shape as counterfactuals. (optional)

    Returns
    -------

    """
    # get indices of unsuccessful counterfactuals
    nan_idx = counterfactuals.index[counterfactuals.isnull().any(axis=1)]
    output_counterfactuals = counterfactuals.copy()
    output_counterfactuals = output_counterfactuals.drop(index=nan_idx)

    if factuals is not None:
        if factuals.shape[0] != counterfactuals.shape[0]:
            raise ValueError(
                "Counterfactuals and factuals should contain the same amount of samples"
            )
        output_factuals = factuals.copy()
        output_factuals = output_factuals.drop(index=nan_idx)
        return output_counterfactuals, output_factuals

    return output_counterfactuals

In [265]:
class Distance(Evaluation):
    """
    Calculates the L0, L1, L2, and L-infty distance measures.
    """

    def __init__(self, mlmodel):
        super().__init__(mlmodel)
        # self.columns = ["L0_distance", "L1_distance", "L2_distance", "Linf_distance"]
        self.columns = ["L2_distance"]
    def get_evaluation(self, factuals, counterfactuals):
        # only keep the rows for which counterfactuals could be found
        counterfactuals_without_nans, factuals_without_nans = remove_nans(
            counterfactuals, factuals
        )

        # return empty dataframe if no successful counterfactuals
        if counterfactuals_without_nans.empty:
            return pd.DataFrame(columns=self.columns)

        arr_f = factuals_without_nans.to_numpy(dtype=np.float64)
        arr_cf = counterfactuals_without_nans.to_numpy(dtype=np.float64)

        distances = _get_distances(arr_f, arr_cf)

        return pd.DataFrame(distances, columns=self.columns)

In [202]:
factuals = factual.copy()
counterfactuals = subset.copy()

In [203]:
factuals

past_profitability_21d     -0.073829
past_profitability_63d      0.213770
past_profitability_126d     0.258345
volatility_21d              0.310263
volatility_63d              0.433814
volatility_126d             0.405843
avg_price_21d               4.959745
avg_price_63d               4.716178
avg_price_126d              4.147089
sharpe_21d                 -0.236372
sharpe_63d                  0.493709
sharpe_126d                 0.637240
m_21d                      -0.375000
m_63d                       0.825000
m_126d                      0.960000
roc_21d                    -0.073829
roc_63d                     0.213770
roc_126d                    0.258345
MACD                       -0.033728
rsi_14                     36.635150
dco_22                      0.123516
min_21d                     4.614000
min_63d                     3.913000
min_126d                    3.320000
max_21d                     5.190000
max_63d                     5.190000
max_126d                    5.190000
e

In [204]:
counterfactuals

,past_profitability_21d,past_profitability_63d,past_profitability_126d,volatility_21d,volatility_63d,volatility_126d,avg_price_21d,avg_price_63d,avg_price_126d,sharpe_21d,...,min_63d,min_126d,max_21d,max_63d,max_126d,exp_mean_21d,exp_mean_63d,exp_mean_126d,target,factual_id
0,0.0,0.3,0.2,0.4,0.4,0.4,5.0,4.6,4.1,0.1,...,3.7,3.3,5.2,5.2,5.2,5.0,4.63,4.19,-0.061932,0


In [267]:
factuals = testing_data[0:1].astype(np.float32)
counterfactuals = dice_exp.cf_examples_list[0].final_cfs_df


In [268]:
factuals

,past_profitability_21d,past_profitability_63d,past_profitability_126d,volatility_21d,volatility_63d,volatility_126d,avg_price_21d,avg_price_63d,avg_price_126d,sharpe_21d,...,min_21d,min_63d,min_126d,max_21d,max_63d,max_126d,exp_mean_21d,exp_mean_63d,exp_mean_126d,target
0,-0.073829,0.21377,0.258345,0.310263,0.433814,0.405843,4.959745,4.716178,4.147089,-0.236372,...,4.614,3.913,3.32,5.19,5.19,5.19,4.868377,4.646188,4.232882,-0.016848


In [269]:
counterfactuals

,past_profitability_21d,past_profitability_63d,past_profitability_126d,volatility_21d,volatility_63d,volatility_126d,avg_price_21d,avg_price_63d,avg_price_126d,sharpe_21d,...,min_21d,min_63d,min_126d,max_21d,max_63d,max_126d,exp_mean_21d,exp_mean_63d,exp_mean_126d,target
0,-0.0,0.1,0.0,0.3,0.3,0.4,2.6,2.7,2.6,-0.1,...,2.5,2.4,2.3,2.7,2.9,2.9,2.6,2.64,2.59,0.201366


In [270]:
# Initialize distance evaluator
dist_eval = Distance(dice_model)


In [271]:
# Compute distances
distance_df = dist_eval.get_evaluation(factuals, counterfactuals)

print(distance_df)

   L2_distance
0   425.531603


In [187]:
len(counterfactuals.columns)

31

In [276]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(training_data.drop(columns="target"))

distances = []
all_cf = []

n = len(testing_data.iloc[:500,:])

for i in range(n):
    query_instance = testing_data.drop(columns="target").iloc[i:i+1].astype(np.float64)
    factual_id = query_instance.index[0]
    # Standardize factual
    factual_scaled = pd.DataFrame(
        scaler.transform(query_instance),
        columns=query_instance.columns,
        index=query_instance.index
    )
    
    dice_exp = exp.generate_counterfactuals(
        query_instance,
        total_CFs=1,
        desired_range=[min, max]
    )
    #cf_df = dice_exp.cf_examples_list[0].final_cfs_df.copy()
    counterfactuals = dice_exp.cf_examples_list[0].final_cfs_df.copy()
    query_instance["target"] = testing_data["target"].iloc[i:i+1].values.astype(np.float64)

    # Standardize counterfactual
    counterfactual_scaled = pd.DataFrame(
        scaler.transform(counterfactuals.drop(columns="target")),
        columns=counterfactuals.columns[:-1],
        index=counterfactuals.index
    )

    # factuals_repeated = pd.concat([query_instance]*len(counterfactuals), ignore_index=True)
    distance = dist_eval.get_evaluation(factual_scaled, counterfactual_scaled)
    # distance_df["factual_id"] = i
    distances.append(distance)

#all_distances = pd.concat(distances, ignore_index=True)
all_distances = np.array(distances)


100%|██████████| 1/1 [00:00<00:00,  1.31it/s]


In [ ]:
all_distances

array([[[8.38933129e-01]],

       [[1.44214311e+00]],

       [[1.30913043e+00]],

       [[2.22260408e+00]],

       [[8.32878675e-01]],

       [[1.98419378e+00]],

       [[1.71965440e+00]],

       [[1.79621091e+00]],

       [[2.00786441e+00]],

       [[1.82573922e+00]],

       [[1.83098330e+00]],

       [[1.19747917e+00]],

       [[1.87910908e+00]],

       [[7.93275835e-01]],

       [[1.65662009e+00]],

       [[1.28022364e+00]],

       [[9.98639963e-01]],

       [[1.56600446e+00]],

       [[2.11728349e+00]],

       [[1.88999309e+00]],

       [[1.97482673e+00]],

       [[2.06185519e+00]],

       [[1.56393465e+00]],

       [[1.86612507e+00]],

       [[1.93544537e+00]],

       [[1.74783966e+00]],

       [[1.69155953e+00]],

       [[2.03656588e+00]],

       [[1.87292843e+00]],

       [[1.25446433e+00]],

       [[1.11636208e+00]],

       [[2.03016849e+00]],

       [[2.16728956e+00]],

       [[1.59906092e+00]],

       [[1.17853952e+00]],

       [[2.57304919e

In [ ]:
np.mean(all_distances)

np.float64(66.11436894793613)

## CARLA

For classification problem! It also has evaluation metrics

In [ ]:
from carla.data.catalog import CsvCatalog

#continuous = ["age", "fnlwgt", "education-num", "capital-gain", "hours-per-week", "capital-loss"]
#categorical = ["marital-status", "native-country", "occupation", "race", "relationship", "sex", "workclass"]
#immutable = ["age", "sex"]
file_path = "training_data.csv" # Path to your data

dataset = CsvCatalog(file_path=file_path,
                    continuous=continuous,
                    categorical=[],
                    immutables=[],
                    target='target')

display(dataset.df.head())

,past_profitability_21d,past_profitability_63d,past_profitability_126d,volatility_21d,volatility_63d,...,max_126d,exp_mean_21d,exp_mean_63d,exp_mean_126d,target
0,0.136332,0.169568,0.211132,0.008955,0.017334,...,0.0011,0.000881,0.000898,0.000927,0.048596
1,0.135998,0.173294,0.212602,0.009270,0.017363,...,0.0011,0.000884,0.000898,0.000927,0.055726
2,0.133644,0.172705,0.210647,0.009641,0.017483,...,0.0011,0.000885,0.000898,0.000926,0.068956
3,0.130236,0.171356,0.204992,0.010008,0.017608,...,0.0011,0.000885,0.000898,0.000925,0.087021
4,0.127791,0.168613,0.199268,0.010343,0.017683,...,0.0011,0.000884,0.000897,0.000924,0.116883


In [3]:
# load catalog model
model_type = "forest"
ml_model = MLModelCatalog(
    dataset,
    model_type=model_type,
    load_online=True,
    backend="sklearn"
)

In [4]:


# define your recourse method
recourse_method = recourse_catalog.Dice(ml_model, hyperparams={})


In [5]:
# get some negative instances
factuals = predict_negative_instances(ml_model, dataset.df)
factuals = factuals[:5]

# find counterfactuals
counterfactuals = recourse_method.get_counterfactuals(factuals)

In [6]:
counterfactuals

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
0,0.301370,0.044131,0.800000,0.02174,0.0,0.459799
1,0.452055,0.048052,0.800000,0.00000,0.0,0.946284
2,0.287671,0.137581,0.858199,0.00000,0.0,0.936285
3,0.493151,0.150486,0.952120,0.00000,0.0,0.823724
4,0.150685,0.220635,0.800000,0.70000,0.0,0.397959


In [7]:
factuals

,age,fnlwgt,education-num,capital-gain,capital-loss,...,occupation_Other,race_White,relationship_Non-Husband,sex_Male,workclass_Private
0,0.301370,0.044131,0.800000,0.02174,0.0,...,0.0,1.0,1.0,1.0,0.0
1,0.452055,0.048052,0.800000,0.00000,0.0,...,0.0,1.0,0.0,1.0,0.0
2,0.287671,0.137581,0.533333,0.00000,0.0,...,1.0,1.0,1.0,1.0,1.0
3,0.493151,0.150486,0.400000,0.00000,0.0,...,1.0,0.0,0.0,1.0,1.0
4,0.150685,0.220635,0.800000,0.00000,0.0,...,0.0,0.0,1.0,0.0,1.0
